In [189]:
import pandas as pd
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

Keep only labels for stress and neutral

In [190]:
features = pd.read_csv("../Data/wesad_eda/feature_dataframe.csv")
neutral = features[features['label'] == 1]
stress = features[features['label'] == 2]
feature_binary = pd.concat([neutral, stress])
feature_binary.columns

Index(['label', 'subject_id', 'phasic_mean', 'phasic_skew',
       'phasic_peak_count', 'phasic_std_rise_time', 'tonic_mean', 'tonic_max'],
      dtype='object')

Apply feature wise per subject normalization

In [191]:
feature_cols = ['phasic_mean', 'phasic_peak_count', 'phasic_std_rise_time', 'tonic_mean', 'tonic_max', 'phasic_skew']

def minmax_per_subject(group):
    return (group - group.min()) / (group.max() - group.min())

feature_binary[feature_cols] = (
    feature_binary
    .groupby('subject_id')[feature_cols]
    .transform(minmax_per_subject)
)

In [192]:
feature_binary = feature_binary.dropna()
#feature_binary = feature_binary.drop(columns = ['phasic_std_rise_time'])
y = feature_binary[['label']].values.ravel()
X = feature_binary.iloc[:, 2:]


Reduced dimension is from keeping only rows with select labels.

In [193]:
print(X.shape)
print(y.shape)

(849, 6)
(849,)


Run Adaboost

In [194]:
groups = feature_binary['subject_id']

logo = LeaveOneGroupOut()

param_grid = {
    'n_estimators': [200, 250, 350, 400],
    'learning_rate': [0.1, 0.05, 0.001, 0.005],
    'estimator__max_depth': [1, 2, 3], 
}

grid = GridSearchCV(
    estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier()),
    param_grid=param_grid,
    cv=logo,
    scoring='accuracy')

grid.fit(X, y, groups=groups)  # groups= is required here, easy to forget

print(f"Best params: {grid.best_params_}")
print(f"Best LOGO accuracy: {grid.best_score_:.4f}")

Best params: {'estimator__max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 400}
Best LOGO accuracy: 0.9037


In [195]:
model = AdaBoostClassifier(n_estimators=50, learning_rate=1)
logo = LeaveOneGroupOut()
# Run the Cross-Validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    groups=groups, 
    cv=logo, 
    scoring='accuracy'
)

# 4. Review the results
print(f"Number of subjects (folds): {len(scores)}")
print(f"Accuracy for each subject: {scores}")
print(f"Average Accuracy across all unseen subjects: {scores.mean():.4f}")

Number of subjects (folds): 15
Accuracy for each subject: [0.88135593 0.70175439 0.92982456 0.80701754 1.         0.92982456
 0.60344828 0.83636364 0.69090909 1.         1.         1.
 0.98214286 0.98245614 1.        ]
Average Accuracy across all unseen subjects: 0.8897


In [196]:
# Initialize Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
# Run Cross-Validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    groups=groups, 
    cv=logo, 
    scoring='accuracy'
)

# Review the results
print(f"Number of subjects (folds): {len(scores)}")
print(f"Accuracy for each subject: {scores}")
print(f"Average Accuracy across all unseen subjects: {scores.mean():.4f}")

Number of subjects (folds): 15
Accuracy for each subject: [0.88135593 0.73684211 0.98245614 0.68421053 0.96491228 0.94736842
 0.60344828 0.83636364 0.70909091 1.         0.94736842 1.
 0.96428571 0.98245614 1.        ]
Average Accuracy across all unseen subjects: 0.8827


In [197]:
# Initialize Support Vector Classifier (SVC)
model = SVC(kernel='rbf', C=1.0, random_state=42)

# Set up Leave-One-Group-Out cross-validation
logo = LeaveOneGroupOut()

# Run Cross-Validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    groups=groups, 
    cv=logo, 
    scoring='accuracy'
)

# Review the results
print(f"Number of subjects (folds): {len(scores)}")
print(f"Accuracy for each subject: {scores}")
print(f"Average Accuracy across all unseen subjects: {scores.mean():.4f}")

Number of subjects (folds): 15
Accuracy for each subject: [0.89830508 0.73684211 1.         0.80701754 1.         1.
 0.53448276 0.85454545 0.69090909 1.         0.96491228 1.
 1.         0.98245614 1.        ]
Average Accuracy across all unseen subjects: 0.8980
